# 02 — Model baseline dan audit fairness (RQ1)

**Outcome utama:** diabetes belum terdiagnosis (HbA1c ≥ 6,5%).
**Outcome sekunder:** disglikemia belum terdiagnosis (HbA1c ≥ 5,7%) — untuk analisis interseksional yang lebih stabil.

Alur:
1. Siapkan data dan prediktor non-laboratorium (ras/etnis dan pendapatan **tidak** dipakai sebagai prediktor, hanya untuk audit).
2. *Nested cross-validation* 5 lipatan: tuning hyperparameter di lipatan dalam, threshold dipilih agar sensitivitas berbobot ≈ 80% pada data latih.
3. Semua peserta mendapat prediksi *out-of-fold*, sehingga ke-647 kasus diabetes ikut diaudit.
4. Metrik berbobot survei dengan interval kepercayaan 95% dari *survey bootstrap* (Rao–Wu, resampling PSU dalam strata).
5. Audit per atribut tunggal, kesenjangan fairness, dan audit interseksional (ras/etnis × usia).

Sebelum menjalankan penuh, boleh set `CEPAT = True` untuk uji kilat (±5 menit). Hasil final pakai `CEPAT = False`.

In [ ]:
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import loguniform

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent  # repository root (run notebooks from notebooks/)
DATA = ROOT / "data" / "processed" / "nhanes_dm_belum_terdiagnosis.csv"
HASIL = ROOT / "hasil"
GAMBAR = HASIL / "gambar"
GAMBAR.mkdir(parents=True, exist_ok=True)

CEPAT = False              # True = uji kilat (tuning dan bootstrap dikurangi)
RANDOM_STATE = 42
N_LIPATAN = 5              # lipatan luar
N_ITER_TUNING = 4 if CEPAT else 15
N_BOOT = 100 if CEPAT else 500
TARGET_SENSITIVITAS = 0.80
MIN_KASUS = 30             # subkelompok dengan < 30 kasus positif ditandai, kesenjangan tidak dihitung

try:
    from xgboost import XGBClassifier
    ADA_XGB = True
except ImportError:
    ADA_XGB = False
try:
    from lightgbm import LGBMClassifier
    ADA_LGBM = True
except ImportError:
    ADA_LGBM = False
print("XGBoost:", ADA_XGB, "| LightGBM:", ADA_LGBM, "| mode cepat:", CEPAT)

## 1. Data dan prediktor
Nilai tekanan darah yang tidak masuk akal (diastolik < 30 atau sistolik < 60 mmHg; NHANES menyimpan nilai 0 sebagai angka sangat kecil) dijadikan data hilang.

In [ ]:
d = pd.read_csv(DATA)
d.loc[d["DBP"] < 30, "DBP"] = np.nan
d.loc[d["SBP"] < 60, "SBP"] = np.nan
d["PEREMPUAN"] = (d["JENIS_KELAMIN"] == "Perempuan").astype(int)

OUTCOME = {
    "DM": ("DM_BELUM_TERDIAGNOSIS", "Diabetes belum terdiagnosis (HbA1c ≥ 6,5%)"),
    "DISGLIKEMIA": ("Y_DISGLIKEMIA", "Disglikemia belum terdiagnosis (HbA1c ≥ 5,7%)"),
}
d["Y_DISGLIKEMIA"] = (d["HBA1C"] >= 5.7).astype(int)

PREDIKTOR = ["USIA", "PEREMPUAN", "BMI", "LINGKAR_PINGGANG", "SBP", "DBP", "RIWAYAT_HIPERTENSI",
             "RIWAYAT_KELUARGA_DM", "PERNAH_MEROKOK", "AKTIF_FISIK", "PENDIDIKAN"]

ATRIBUT = {"KEL_USIA": "Usia", "JENIS_KELAMIN": "Jenis kelamin",
           "RAS_ETNIS": "Ras/etnis", "KEL_PENDAPATAN": "Pendapatan (PIR)"}
d["RAS_X_USIA"] = d["RAS_ETNIS"] + " | " + d["KEL_USIA"].astype(str)

X = d[PREDIKTOR].astype(float).values
W = d["WT_GABUNGAN"].values
print(f"N = {len(d):,}")
for k, (kol, ket) in OUTCOME.items():
    print(f"{ket}: {int(d[kol].sum()):,} kasus "
          f"(prevalensi berbobot {100 * np.average(d[kol], weights=W):.2f}%)")

## 2. Model dan ruang tuning
Regresi logistik, Random Forest, XGBoost, dan LightGBM. Ketidakseimbangan kelas ditangani dengan bobot kelas (tanpa SMOTE). Kalau XGBoost/LightGBM belum terpasang, dipakai HistGradientBoosting sebagai pengganti.

In [ ]:
def daftar_model():
    m = {
        "Regresi logistik": (
            Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                      ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))]),
            {"clf__C": loguniform(1e-3, 10)}),
        "Random Forest": (
            Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                     n_jobs=-1, random_state=RANDOM_STATE))]),
            {"clf__max_depth": [None, 6, 10, 16], "clf__min_samples_leaf": [5, 10, 20, 50],
             "clf__max_features": ["sqrt", 0.5]}),
    }
    if ADA_XGB:
        m["XGBoost"] = (
            XGBClassifier(n_estimators=400, tree_method="hist", eval_metric="logloss",
                          n_jobs=-1, random_state=RANDOM_STATE),
            {"max_depth": [3, 4, 6], "learning_rate": [0.03, 0.05, 0.1], "subsample": [0.7, 0.9, 1.0],
             "colsample_bytree": [0.7, 0.9, 1.0], "min_child_weight": [1, 5, 10]})
    if ADA_LGBM:
        m["LightGBM"] = (
            LGBMClassifier(n_estimators=400, is_unbalance=True, subsample_freq=1,
                           random_state=RANDOM_STATE, verbose=-1),
            {"num_leaves": [15, 31, 63], "learning_rate": [0.03, 0.05, 0.1],
             "min_child_samples": [20, 50, 100], "subsample": [0.7, 0.9, 1.0],
             "colsample_bytree": [0.7, 0.9, 1.0]})
    if not (ADA_XGB or ADA_LGBM):
        m["HistGradientBoosting"] = (
            HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
            {"max_depth": [3, 5, None], "learning_rate": [0.03, 0.05, 0.1], "min_samples_leaf": [20, 50, 100]})
    return m

MODEL = daftar_model()
list(MODEL)

## 3. Nested cross-validation dan prediksi out-of-fold
Di setiap lipatan luar: tuning (3 lipatan dalam, skor AUPRC) → rekalibrasi Platt dari prediksi out-of-fold data latih (bobot kelas membuat probabilitas terlalu tinggi; rekalibrasi tidak mengubah AUROC) → threshold dipilih agar sensitivitas berbobot ≥ 80% → prediksi untuk lipatan uji. Bagian ini paling lama (puluhan menit untuk mode penuh).

In [ ]:
def logit(q):
    q = np.clip(q, 1e-6, 1 - 1e-6)
    return np.log(q / (1 - q))


def pilih_threshold(y, p, w, target=TARGET_SENSITIVITAS):
    # threshold tertinggi yang menghasilkan sensitivitas berbobot >= target
    pos = y == 1
    urut = np.argsort(-p[pos])
    p_pos, w_pos = p[pos][urut], w[pos][urut]
    kum = np.cumsum(w_pos) / w_pos.sum()
    return p_pos[np.searchsorted(kum, target)]


def jalankan_nested_cv(y):
    strat = d["SIKLUS"].astype(str) + "_" + y.astype(str)
    luar = StratifiedKFold(n_splits=N_LIPATAN, shuffle=True, random_state=RANDOM_STATE)
    hasil = []
    for nama, (est, ruang) in MODEL.items():
        mulai = time.time()
        prob = np.zeros(len(y))
        pred = np.zeros(len(y), dtype=int)
        for lipat, (tr, te) in enumerate(luar.split(X, strat)):
            model = clone(est)
            if nama == "XGBoost":
                model.set_params(scale_pos_weight=(y[tr] == 0).sum() / (y[tr] == 1).sum())
            cari = RandomizedSearchCV(model, ruang, n_iter=N_ITER_TUNING, scoring="average_precision",
                                      cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                                      random_state=RANDOM_STATE, n_jobs=1)
            cari.fit(X[tr], y[tr])
            terbaik = cari.best_estimator_
            oof_latih = cross_val_predict(clone(terbaik), X[tr], y[tr], method="predict_proba",
                                          cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE))[:, 1]
            # rekalibrasi Platt: bobot kelas membuat probabilitas terlalu tinggi
            platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(oof_latih).reshape(-1, 1), y[tr],
                                                                 sample_weight=W[tr] / W[tr].mean())
            kal = lambda q: platt.predict_proba(logit(q).reshape(-1, 1))[:, 1]
            thr = pilih_threshold(y[tr], kal(oof_latih), W[tr])
            prob[te] = kal(terbaik.predict_proba(X[te])[:, 1])
            pred[te] = (prob[te] >= thr).astype(int)
            hasil.append(pd.DataFrame({"SEQN": d["SEQN"].values[te], "MODEL": nama, "LIPATAN": lipat,
                                       "PROB": prob[te], "PRED": pred[te], "THRESHOLD": thr}))
        print(f"   {nama:22s} selesai ({(time.time() - mulai) / 60:.1f} menit)")
    return pd.concat(hasil, ignore_index=True)


prediksi = {}
for kode, (kol, ket) in OUTCOME.items():
    print(ket)
    y = d[kol].values.astype(int)
    p = jalankan_nested_cv(y)
    p["OUTCOME"] = kode
    p["Y"] = p["SEQN"].map(dict(zip(d["SEQN"], y)))
    prediksi[kode] = p

semua_prediksi = pd.concat(prediksi.values(), ignore_index=True)
semua_prediksi.to_csv(ROOT / "data" / "processed" / "prediksi_oof_02.csv", index=False)
print("Prediksi out-of-fold disimpan.")

## 4. Metrik berbobot dan survey bootstrap
Bobot replikasi Rao–Wu: di tiap strata dengan *n* PSU, diambil *n − 1* PSU dengan pengembalian; bobot dikali *n/(n−1)* × jumlah terpilih. Semua metrik (AUROC, AUPRC, sensitivitas, spesifisitas, PPV, FNR, FPR, Brier, rasio observasi/ekspektasi) dihitung dengan bobot survei.

In [ ]:
def bobot_replikasi(n_boot, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    psu = d["SDMVSTRA"].astype(int).astype(str) + "_" + d["SDMVPSU"].astype(int).astype(str)
    faktor = np.zeros((len(d), n_boot), dtype=np.float32)
    for strata, g in d.groupby("SDMVSTRA"):
        daftar = psu[g.index].unique()
        n_h = len(daftar)
        if n_h < 2:
            faktor[g.index, :] = 1.0
            continue
        for b in range(n_boot):
            ambil = rng.choice(daftar, size=n_h - 1, replace=True)
            hitung = pd.Series(ambil).value_counts()
            f = psu[g.index].map(hitung).fillna(0).values * n_h / (n_h - 1)
            faktor[g.index, b] = f
    return faktor


def metrik(y, p, yhat, w):
    out = {}
    if w.sum() == 0:
        return {k: np.nan for k in ["AUROC", "AUPRC", "Sensitivitas", "Spesifisitas", "PPV", "FNR", "FPR", "Brier", "O/E"]}
    pos, neg = y == 1, y == 0
    dua_kelas = w[pos].sum() > 0 and w[neg].sum() > 0
    out["AUROC"] = roc_auc_score(y, p, sample_weight=w) if dua_kelas else np.nan
    out["AUPRC"] = average_precision_score(y, p, sample_weight=w) if dua_kelas else np.nan
    tp = w[pos & (yhat == 1)].sum(); fn = w[pos & (yhat == 0)].sum()
    tn = w[neg & (yhat == 0)].sum(); fp = w[neg & (yhat == 1)].sum()
    out["Sensitivitas"] = tp / (tp + fn) if tp + fn > 0 else np.nan
    out["Spesifisitas"] = tn / (tn + fp) if tn + fp > 0 else np.nan
    out["PPV"] = tp / (tp + fp) if tp + fp > 0 else np.nan
    out["FNR"] = 1 - out["Sensitivitas"]
    out["FPR"] = 1 - out["Spesifisitas"]
    out["Brier"] = np.average((p - y) ** 2, weights=w)
    out["O/E"] = np.average(y, weights=w) / np.average(p, weights=w)
    return out


def metrik_dengan_ci(y, p, yhat, w, faktor):
    titik = metrik(y, p, yhat, w)
    rep = pd.DataFrame([metrik(y, p, yhat, w * faktor[:, b]) for b in range(faktor.shape[1])])
    baris = {}
    for k, v in titik.items():
        baris[k] = v
        baris[k + "_lo"] = np.nanpercentile(rep[k], 2.5)
        baris[k + "_hi"] = np.nanpercentile(rep[k], 97.5)
    return baris, rep


FAKTOR = bobot_replikasi(N_BOOT)
print("Bobot replikasi:", FAKTOR.shape)

## 5. Performa keseluruhan
Model terbaik per outcome dipilih berdasarkan AUPRC (cocok untuk prevalensi rendah).

In [ ]:
idx_seqn = pd.Series(np.arange(len(d)), index=d["SEQN"].values)

def susun(kode, nama_model):
    p = prediksi[kode]
    p = p[p["MODEL"] == nama_model].set_index("SEQN").loc[d["SEQN"]]
    return p["Y"].values.astype(int), p["PROB"].values, p["PRED"].values.astype(int)

baris = []
for kode, (kol, ket) in OUTCOME.items():
    for nama in MODEL:
        y, p, yhat = susun(kode, nama)
        m, _ = metrik_dengan_ci(y, p, yhat, W, FAKTOR)
        baris.append({"Outcome": kode, "Model": nama, **m})
performa = pd.DataFrame(baris)
performa.to_csv(HASIL / "02_performa_keseluruhan.csv", index=False)

TERBAIK = performa.loc[performa.groupby("Outcome")["AUPRC"].idxmax()].set_index("Outcome")["Model"].to_dict()
print("Model terbaik (AUPRC):", TERBAIK)

def fmt(r, k, n=3):
    return f"{r[k]:.{n}f} ({r[k + '_lo']:.{n}f}–{r[k + '_hi']:.{n}f})"

tampil = performa.apply(lambda r: pd.Series({
    "Outcome": r["Outcome"], "Model": r["Model"], "AUROC": fmt(r, "AUROC"), "AUPRC": fmt(r, "AUPRC"),
    "Sensitivitas": fmt(r, "Sensitivitas"), "Spesifisitas": fmt(r, "Spesifisitas"),
    "Brier": fmt(r, "Brier", 4), "O/E": fmt(r, "O/E", 2)}), axis=1)
tampil

## 6. Audit per subkelompok (RQ1)
Untuk setiap model, outcome, dan subkelompok: metrik berbobot + CI 95%. Subkelompok dengan < 30 kasus positif diberi tanda dan tidak ikut dalam perhitungan kesenjangan.

In [ ]:
audit, replikasi = [], {}
for kode, (kol, ket) in OUTCOME.items():
    for nama in MODEL:
        y, p, yhat = susun(kode, nama)
        for atr, label_atr in ATRIBUT.items():
            for grup in sorted(d[atr].dropna().astype(str).unique()):
                mask = (d[atr].astype(str) == grup).values
                n_kasus = int(y[mask].sum())
                m, rep = metrik_dengan_ci(y[mask], p[mask], yhat[mask], W[mask], FAKTOR[mask])
                audit.append({"Outcome": kode, "Model": nama, "Atribut": label_atr, "Kelompok": grup,
                              "N": int(mask.sum()), "Kasus": n_kasus, "Cukup_kasus": n_kasus >= MIN_KASUS, **m})
                replikasi[(kode, nama, label_atr, grup)] = rep
audit = pd.DataFrame(audit)
audit.to_csv(HASIL / "02_audit_subkelompok.csv", index=False)

# ringkasan untuk model terbaik
ringkas = audit[audit.apply(lambda r: TERBAIK[r["Outcome"]] == r["Model"], axis=1)]
ringkas[["Outcome", "Atribut", "Kelompok", "N", "Kasus", "AUROC", "Sensitivitas", "FNR", "FPR", "O/E"]].round(3)

## 7. Kesenjangan fairness
Per atribut (hanya kelompok dengan ≥ 30 kasus):
- **Selisih FNR** (equal opportunity): FNR tertinggi − terendah.
- **Equalized odds difference:** maksimum dari selisih TPR dan selisih FPR.
- **Selisih AUROC** dan **rentang O/E** (kalibrasi).
CI 95% dihitung dari replikasi bootstrap yang sama.

In [ ]:
def kesenjangan(rep_per_grup):
    tab = {g: r for g, r in rep_per_grup.items()}
    fnr = pd.concat({g: r["FNR"] for g, r in tab.items()}, axis=1)
    fpr = pd.concat({g: r["FPR"] for g, r in tab.items()}, axis=1)
    auc = pd.concat({g: r["AUROC"] for g, r in tab.items()}, axis=1)
    oe = pd.concat({g: r["O/E"] for g, r in tab.items()}, axis=1)
    sel = lambda m: m.max(axis=1) - m.min(axis=1)
    return pd.DataFrame({"Selisih FNR": sel(fnr), "EOD": np.maximum(sel(fnr), sel(fpr)),
                         "Selisih AUROC": sel(auc), "Rentang O/E": sel(oe)})

baris = []
for kode in OUTCOME:
    for nama in MODEL:
        for atr, label_atr in ATRIBUT.items():
            sub = audit[(audit.Outcome == kode) & (audit.Model == nama) & (audit.Atribut == label_atr) & audit.Cukup_kasus]
            if len(sub) < 2:
                continue
            # estimasi titik dari data penuh
            titik = {"Selisih FNR": sub.FNR.max() - sub.FNR.min(),
                     "EOD": max(sub.FNR.max() - sub.FNR.min(), sub.FPR.max() - sub.FPR.min()),
                     "Selisih AUROC": sub.AUROC.max() - sub.AUROC.min(),
                     "Rentang O/E": sub["O/E"].max() - sub["O/E"].min()}
            rep = kesenjangan({g: replikasi[(kode, nama, label_atr, g)] for g in sub.Kelompok})
            r = {"Outcome": kode, "Model": nama, "Atribut": label_atr, "Kelompok_dibandingkan": len(sub),
                 "FNR_tertinggi": sub.loc[sub.FNR.idxmax(), "Kelompok"], "FNR_terendah": sub.loc[sub.FNR.idxmin(), "Kelompok"]}
            for k, v in titik.items():
                r[k] = v
                r[k + "_lo"] = np.nanpercentile(rep[k], 2.5)
                r[k + "_hi"] = np.nanpercentile(rep[k], 97.5)
            baris.append(r)
gap = pd.DataFrame(baris)
gap.to_csv(HASIL / "02_kesenjangan_fairness.csv", index=False)
gap[gap.apply(lambda r: TERBAIK[r["Outcome"]] == r["Model"], axis=1)].round(3)

## 8. Audit interseksional: ras/etnis × usia (model terbaik)

In [ ]:
inter = []
for kode in OUTCOME:
    nama = TERBAIK[kode]
    y, p, yhat = susun(kode, nama)
    for grup in sorted(d["RAS_X_USIA"].unique()):
        mask = (d["RAS_X_USIA"] == grup).values
        n_kasus = int(y[mask].sum())
        m, _ = metrik_dengan_ci(y[mask], p[mask], yhat[mask], W[mask], FAKTOR[mask])
        ras, usia = grup.split(" | ")
        inter.append({"Outcome": kode, "Model": nama, "Ras/etnis": ras, "Usia": usia, "N": int(mask.sum()),
                      "Kasus": n_kasus, "Cukup_kasus": n_kasus >= MIN_KASUS, **m})
inter = pd.DataFrame(inter)
inter.to_csv(HASIL / "02_audit_interseksional.csv", index=False)
inter.pivot_table(index="Ras/etnis", columns=["Outcome", "Usia"], values="FNR").round(2)

## 9. Gambar untuk naskah
- **Gambar A:** FNR (proporsi kasus yang terlewat) per subkelompok, model terbaik, dua outcome berdampingan. Garis putus-putus = FNR keseluruhan.
- **Gambar B:** heatmap FNR ras/etnis × usia untuk outcome sekunder; sel dengan < 30 kasus diarsir dan tidak diberi angka.

In [ ]:
BIRU = "#2a78d6"; ABU = "#52514e"; TINTA = "#0b0b0b"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#8a8984", "axes.labelcolor": TINTA, "xtick.color": ABU, "ytick.color": ABU})

urutan = []
for atr, label_atr in ATRIBUT.items():
    for g in sorted(d[atr].dropna().astype(str).unique()):
        urutan.append((label_atr, g))

fig, axes = plt.subplots(1, 2, figsize=(10, 0.28 * len(urutan) + 1.6), sharey=True)
for ax, kode in zip(axes, OUTCOME):
    sub = audit[(audit.Outcome == kode) & (audit.Model == TERBAIK[kode])].set_index(["Atribut", "Kelompok"])
    keseluruhan = performa[(performa.Outcome == kode) & (performa.Model == TERBAIK[kode])]["FNR"].iloc[0]
    ys = np.arange(len(urutan))[::-1]
    for yy, key in zip(ys, urutan):
        r = sub.loc[key]
        warna = BIRU if r["Cukup_kasus"] else "#b8b7b1"
        ax.plot([r["FNR_lo"], r["FNR_hi"]], [yy, yy], color=warna, lw=1.5, solid_capstyle="round")
        ax.plot(r["FNR"], yy, "o", ms=6, color=warna, mec="white", mew=1)
    ax.axvline(keseluruhan, color=ABU, lw=1, ls="--")
    ax.set_xlim(0, 1)
    ax.set_xlabel("False negative rate (kasus terlewat)")
    ax.set_title(f"{OUTCOME[kode][1]}\n{TERBAIK[kode]}", fontsize=9, loc="left", color=TINTA)
    ax.grid(axis="x", color="#e6e5e0", lw=0.6)
axes[0].set_yticks(np.arange(len(urutan))[::-1])
axes[0].set_yticklabels([f"{a}: {g}" for a, g in urutan])
fig.text(0.01, 0.005, "Titik = estimasi berbobot survei; garis = CI 95% (survey bootstrap). Abu-abu = < 30 kasus.",
         fontsize=7.5, color=ABU)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(GAMBAR / "02A_fnr_subkelompok.png", dpi=300)
plt.show()

In [ ]:
seq = LinearSegmentedColormap.from_list("biru", ["#cde2fb", "#86b6ef", "#2a78d6", "#1c5cab", "#104281"])
kode = "DISGLIKEMIA"
sub = inter[inter.Outcome == kode]
ras_urut = sorted(sub["Ras/etnis"].unique())
usia_urut = ["20-39", "40-59", "60+"]
nilai = sub.pivot(index="Ras/etnis", columns="Usia", values="FNR").loc[ras_urut, usia_urut]
cukup = sub.pivot(index="Ras/etnis", columns="Usia", values="Cukup_kasus").loc[ras_urut, usia_urut]

fig, ax = plt.subplots(figsize=(6.4, 4.0))
tampil_nilai = nilai.where(cukup)
im = ax.imshow(tampil_nilai.values.astype(float), cmap=seq, aspect="auto",
               vmin=0, vmax=max(0.6, np.nanmax(tampil_nilai.values)))
for i in range(len(ras_urut)):
    for j in range(len(usia_urut)):
        if cukup.iloc[i, j]:
            v = nilai.iloc[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8.5,
                    color="white" if v > 0.35 else TINTA)
        else:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor="#f0efec", edgecolor="#b8b7b1",
                                       hatch="///", lw=0))
            ax.text(j, i, "<30 kasus", ha="center", va="center", fontsize=7, color=ABU)
ax.set_xticks(range(len(usia_urut))); ax.set_xticklabels(usia_urut)
ax.set_yticks(range(len(ras_urut))); ax.set_yticklabels(ras_urut)
ax.set_xlabel("Kelompok usia (tahun)")
ax.set_title(f"FNR interseksional (ras/etnis × usia)\n{OUTCOME[kode][1]} — {TERBAIK[kode]}",
             fontsize=9, loc="left")
for s in ax.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, ax=ax, shrink=0.8); cb.set_label("False negative rate")
fig.tight_layout()
fig.savefig(GAMBAR / "02B_heatmap_interseksional.png", dpi=300)
plt.show()

## Hasil yang tersimpan
| File | Isi |
| --- | --- |
| `hasil\02_performa_keseluruhan.csv` | Performa semua model, dua outcome, dengan CI 95% |
| `hasil\02_audit_subkelompok.csv` | Metrik per subkelompok, semua model |
| `hasil\02_kesenjangan_fairness.csv` | Selisih FNR, EOD, selisih AUROC, rentang O/E per atribut |
| `hasil\02_audit_interseksional.csv` | Ras/etnis × usia, model terbaik |
| `hasil\gambar\02A_fnr_subkelompok.png`, `02B_heatmap_interseksional.png` | Gambar naskah (300 dpi) |
| `data\processed\prediksi_oof_02.csv` | Prediksi out-of-fold (dipakai notebook 03) |

Langkah berikutnya: notebook **03 — mitigasi bias (RQ2) dan SHAP per kelompok (RQ3)**.